In [1]:
%config Completer.use_jedi = False

In [2]:
# load module
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorboard as tb 
import pytorch_lightning as pl
from itertools import product

from pathlib import Path
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger


from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer ,MultiNormalizer 
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from IPython.core.display import HTML

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']

from my_funs_nei import data_processing , moving_average_alpha_both

/home/nplab/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /home/nplab/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will n

In [3]:
factor = 0.3
train_data = data_processing('nei_train.csv' , factor, moving_average_alpha_both)
train_data

,REG_DTIME,h_dong,count,pops,windspd,humid,temp,precip_form,precip,isHoliday,nei1,nei_count,DOW,HOD,MOY,time_idx
0,2016-01-01 01:00:00,석사동,3.844368e-39,39299.0,1.0,77.55,1.685,2.0,1.0,True,1.0,2.099790,4,1,1,0
1,2016-01-01 01:00:00,신동면,1.406015e-70,2527.0,1.0,79.15,1.640,0.0,0.0,True,1.0,1.723997,4,1,1,0
2,2016-01-01 01:00:00,강남동,6.066180e-07,16830.0,1.0,79.15,1.740,0.0,0.0,True,1.0,2.020122,4,1,1,0
3,2016-01-01 01:00:00,서 면,7.669700e-24,4033.0,1.8,80.25,1.915,2.0,1.0,True,0.0,0.184700,4,1,1,0
4,2016-01-01 01:00:00,남 면,8.142068e-04,1095.0,1.8,83.70,3.340,0.0,0.0,True,0.0,0.000702,4,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1104742,2021-12-31 23:00:00,근화동,1.632297e-15,8540.0,0.7,72.00,-12.400,0.0,0.0,False,2.0,4.235418,4,23,12,52558
1104743,2021-12-31 23:00:00,동 면,1.800822e-06,19649.0,0.7,72.00,-12.400,0.0,0.0,False,0.0,0.333701,4,23,12,52558
1104744,2021-12-31 23:00:00,신사우동,2.633051e-04,22935.0,0.7,72.00,-12.400,0.0,0.0,False,0.0,0.378911,4,23,12,52558
1104745,2021-12-31 23:00:00,약사명동,1.295569e-06,3278.0,0.7,72.00,-12.400,0.0,0.0,False,2.0,2.923618,4,23,12,52558


In [4]:
def get_training_mulit(data , max_prediction_length = 24 , max_encoder_length = 24*7):
    # traing data 생성
    training_cutoff = data["time_idx"].max() - max_prediction_length
    training = TimeSeriesDataSet(
        data[lambda x : x.time_idx <= training_cutoff],
        time_idx = "time_idx",
        target = ["count", 'nei_count'],
        group_ids = ['h_dong'],
        min_encoder_length=max_encoder_length // 2,  # keep encoder length long (as it is in the validation set)
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=24,
        static_categoricals=["h_dong"],
        time_varying_known_categoricals=["HOD", "DOW" , 'isHoliday', 'MOY'],
        time_varying_known_reals=['pops'],
        time_varying_unknown_categoricals=['precip_form'],
        time_varying_unknown_reals=['count','windspd' , 'temp' ,'precip', 'nei1'], 
        target_normalizer=MultiNormalizer([
        GroupNormalizer(
            groups=["h_dong"], 
            transformation="relu",
            center = False
        ),
        GroupNormalizer(
            groups=["h_dong"], 
            transformation="relu",
            center = False
        )]),
        add_relative_time_idx=True,
        add_target_scales=False,
        add_encoder_length=True,
        #allow_missing=True,
        allow_missing_timesteps = True,
        #predict_mode = False
        )
    return training

In [5]:
training = get_training_mulit(train_data,24, 24*7)
training

TimeSeriesDataSet[length=1105734](
	time_idx='time_idx',
	target=['count', 'nei_count'],
	group_ids=['h_dong'],
	weight=None,
	max_encoder_length=168,
	min_encoder_length=84,
	min_prediction_idx=0,
	min_prediction_length=1,
	max_prediction_length=24,
	static_categoricals=['h_dong'],
	static_reals=['encoder_length'],
	time_varying_known_categoricals=['HOD', 'DOW', 'isHoliday', 'MOY'],
	time_varying_known_reals=['pops', 'relative_time_idx'],
	time_varying_unknown_categoricals=['precip_form'],
	time_varying_unknown_reals=['count', 'windspd', 'temp', 'precip', 'nei1'],
	variable_groups={},
	constant_fill_strategy={},
	allow_missing_timesteps=True,
	lags={},
	add_relative_time_idx=True,
	add_target_scales=False,
	add_encoder_length=True,
	target_normalizer=MultiNormalizer(
	normalizers=[GroupNormalizer(
	method='standard',
	groups=['h_dong'],
	center=False,
	scale_by_group=False,
	transformation='relu',
	method_kwargs={}
), GroupNormalizer(
	method='standard',
	groups=['h_dong'],
	center=Fa

In [6]:
hidden_size = 32
atten_head = 4
batch_size= 32
pl.seed_everything(42)
factor = 0.3

train_data = data_processing('nei_train.csv' , factor, moving_average_alpha_both)
training = get_training_mulit(train_data,24, 24*7)

dataloader = training.to_dataloader(batch_size)
validation = TimeSeriesDataSet.from_dataset(training, train_data, predict=True, stop_randomization=True)
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size*100, num_workers=0)

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
lr_logger = LearningRateMonitor()  
logger = TensorBoardLogger('ckpt')
trainer = pl.Trainer(
    max_epochs=300,
    gpus=1,
    gradient_clip_val=0.1,
    limit_train_batches=20,  
    callbacks=[lr_logger],
    logger=logger,
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.01,
    hidden_size=hidden_size,  
    attention_head_size=atten_head,
    dropout=0.1,
    hidden_continuous_size=hidden_size,
    output_size=[7,7],
    loss=QuantileLoss(),
    reduce_on_plateau_patience=4,
)

trainer.fit(tft, 
train_dataloaders=train_dataloader,
val_dataloaders=val_dataloader)


print(trainer.checkpoint_callback.best_model_path)

Global seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | MultiLoss                       | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 538   
3  | prescalers                         | ModuleDict                      | 512   
4  | static_variable_selection          | VariableSelectionNetwork        | 4.6 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 34.9 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 10.0 K
7  | static_context_variable_s

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=300` reached.


ckpt/lightning_logs/version_2/checkpoints/epoch=299-step=6000.ckpt
